# Triumph: adapt the needle reader to real dashboard appearance

This new experiment fine-tunes the actual needle model. It starts from the approved epoch-34 combined model's separate needle branch; that branch was frozen in the previous experiment and will now learn.

**512×512 grayscale; LCD pixels masked to 128.** The public data have 400 synthetic training images from source plates 0/180 and 100 validation images from plate 204. These plates share a recording and donor pixels, so this is a development holdout, not independent real-camera accuracy. A private approved bundle adds 10 real training frames and 9 separate same-recording diagnostic frames; one low-confidence diagnostic frame is excluded from speed metrics. The private images and weights stay in Drive.

Choose a **T4 GPU**, then **Run all**. The default is 25 epochs. Epoch zero, every completed epoch, best weights and resumable state are saved to Drive. There is one training call. No 98% accuracy promise is made.


In [ ]:
#@title Experiment settings
TRAIN_MODE = "transfer" #@param ["transfer", "resume_latest", "resume_best", "fresh"]
EXPERIMENT_NAME = "speedometer_real_needle_v3" #@param {type:"string"}
EPOCHS = 25 #@param {type:"integer"}
BATCH_SIZE = 8 #@param {type:"integer"}
DECODER_LR = 0.0001 #@param {type:"number"}
ENCODER_LR = 0.00002 #@param {type:"number"}
SOURCE_CHECKPOINT = "/content/drive/MyDrive/SpeedometerTraining/experiments/speedometer_arc_v2_augmented/runs/20261017_120108_959025_fee960/best_model.pt" #@param {type:"string"}
PRIVATE_ZIP_PATH = "/content/drive/MyDrive/SpeedometerTraining/inputs/private_real_data_v3.zip" #@param {type:"string"}
DOWNLOAD_COMPACT_RESULTS = True #@param {type:"boolean"}

# Publication pins: filled after the source and data are published.
CODE_COMMIT = "053ce9692a4471a76e8445d76e61b036e3a4d96c"
PUBLIC_DATA_URL = f"https://raw.githubusercontent.com/sainadh812/triumph-speedometer-dataset/{CODE_COMMIT}/real_adaptation_v3/real_dashboard_lcd_mask_500_v3.zip"
PUBLIC_DATA_SHA256 = "55df4c5ec13830f1eaa9cff7a01f3db1d2d1a03505b0aa3422e1db2c0e51200a"
PRIVATE_DATA_SHA256 = "56a4ce66bc3ebb2904bae8b06be2baa6bdcbd1cdc1862ec61c9b5f8df922b8d5"
SOURCE_SHA256 = "4f56b76d8b5b44a1d0a7250b52ed0b7cb3e37cd7a8baff0bdd21eff1221fa4c1"


## 1. GPU and durable storage
`transfer` creates the named experiment once. If it already exists, select `resume_latest` or `resume_best`; neither silently starts over. `fresh` creates a new uniquely named experiment with random weights. Resuming restores the saved 25-epoch target and settings, rather than adding another 25 epochs. GPU availability can vary on free Colab.


In [ ]:
import os,sys,json,re,hashlib,subprocess,uuid,zipfile
from pathlib import Path
from dataclasses import asdict
import numpy as np
import torch,torchvision,PIL,cv2
from IPython.display import display
if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then Run all. Training has not started.")
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive must be connected before training can save checkpoints.")
DRIVE_ROOT = Path("/content/drive/MyDrive/SpeedometerTraining")
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
probe = DRIVE_ROOT / (".write_probe_"+uuid.uuid4().hex[:12])
probe.write_text("ok"); assert probe.read_text()=="ok"; probe.unlink()
WORK_DIR = Path("/content/speedometer_real_needle_v3_work")
WORK_DIR.mkdir(parents=True,exist_ok=True)
print("GPU:",torch.cuda.get_device_name(0),"| torch:",torch.__version__)
print("Persistent checkpoints:",DRIVE_ROOT / "experiments" / EXPERIMENT_NAME)


## 2. Restore the pinned public implementation
This checks out one exact Git commit. Private images and model weights are never read from GitHub. If the notebook's publication pins are unresolved it stops before downloading data or training.


In [ ]:
for name,value,length in [("CODE_COMMIT",CODE_COMMIT,40),("PUBLIC_DATA_SHA256",PUBLIC_DATA_SHA256,64),("PRIVATE_DATA_SHA256",PRIVATE_DATA_SHA256,64)]:
    if re.fullmatch(rf"[0-9a-f]{{{length}}}",value) is None:
        raise ValueError(name+" has no finalized publication pin; use the published completed notebook.")
if globals().get("_V3_IMPORTED_COMMIT",CODE_COMMIT)!=CODE_COMMIT:
    raise RuntimeError("Code commit changed in this runtime. Restart the runtime before importing another version.")
CODE_DIR = WORK_DIR / ("code_"+CODE_COMMIT[:12])
REPOSITORY_URL = "https://github.com/sainadh812/triumph-speedometer-dataset.git"
if not CODE_DIR.exists():
    subprocess.run(["git","clone","--filter=blob:none","--no-checkout",REPOSITORY_URL,str(CODE_DIR)],check=True)
    subprocess.run(["git","-C",str(CODE_DIR),"fetch","--depth","1","origin",CODE_COMMIT],check=True)
    subprocess.run(["git","-C",str(CODE_DIR),"sparse-checkout","set","--no-cone","/training_pilot/*.py","/real_adaptation_v3/*.py"],check=True)
    subprocess.run(["git","-C",str(CODE_DIR),"checkout","--detach",CODE_COMMIT],check=True)
actual_commit=subprocess.check_output(["git","-C",str(CODE_DIR),"rev-parse","HEAD"],text=True).strip()
if actual_commit!=CODE_COMMIT:
    raise ValueError("Local code checkout differs from the pinned commit")
subprocess.run(["git","-C",str(CODE_DIR),"diff","--exit-code","HEAD","--"],check=True)
for module_name,module in list(sys.modules.items()):
    if module_name.startswith(("real_adaptation_v3", "training_pilot")) and getattr(module,"__file__",None):
        if not Path(module.__file__).resolve().is_relative_to(CODE_DIR.resolve()):
            raise RuntimeError("Another speedometer implementation is already imported. Restart the runtime before Run all.")
sys.path.insert(0,str(CODE_DIR))
from real_adaptation_v3 import notebook_helpers as nh
from real_adaptation_v3 import train_needle as trainer
_V3_IMPORTED_COMMIT=CODE_COMMIT
print("Verified code commit:",CODE_COMMIT)


## 3. Public synthetic data and private approved real frames
The public ZIP is checksum verified. If the private bundle is absent from its specified Drive path, upload **private_real_data_v3.zip** once; its exact checksum is checked and the file is saved to Drive. The bundle is then reused on restart. It is never uploaded to GitHub.


In [ ]:
PUBLIC_ARCHIVE=nh.download_verified(PUBLIC_DATA_URL,WORK_DIR/"real_dashboard_lcd_mask_500_v3.zip",PUBLIC_DATA_SHA256)
DATA_DIR=nh.safe_extract(PUBLIC_ARCHIVE,WORK_DIR/("public_"+PUBLIC_DATA_SHA256[:12]),PUBLIC_DATA_SHA256,"dataset500")
private_archive=Path(PRIVATE_ZIP_PATH)
if not private_archive.is_file():
    from google.colab import files
    print("Upload the approved private_real_data_v3.zip; it will be checked and saved to",private_archive)
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise ValueError("Expected exactly one approved private ZIP")
    uploaded_name,uploaded_bytes=next(iter(uploaded.items()))
    if hashlib.sha256(uploaded_bytes).hexdigest()!=PRIVATE_DATA_SHA256:
        raise ValueError("Private bundle checksum mismatch; nothing was saved to Drive")
    private_archive.parent.mkdir(parents=True,exist_ok=True)
    if private_archive.exists():
        raise FileExistsError("A private bundle appeared at the destination; rerun setup to verify it")
    temporary=private_archive.parent/(".upload_"+uuid.uuid4().hex[:12]+".partial")
    temporary.write_bytes(uploaded_bytes)
    os.replace(temporary,private_archive)
    del uploaded,uploaded_bytes
if nh.sha256(private_archive)!=PRIVATE_DATA_SHA256:
    raise ValueError("The private Drive ZIP differs from the approved bundle")
REAL_DATA_DIR=nh.safe_extract(private_archive,WORK_DIR/("private_"+PRIVATE_DATA_SHA256[:12]),PRIVATE_DATA_SHA256,"real_data")
print("Verified synthetic data:",DATA_DIR)
print("Verified private real data:",REAL_DATA_DIR)


## 4. Verified initialization and restart setup
Transfer extracts the existing needle weights and starts a new optimizer at epoch zero. It fine-tunes the decoder and heads plus the last four encoder blocks. Fresh training also enables the full randomly initialized encoder. The earlier combined model and its checkpoints are preserved.

Validation selects the best checkpoint using needle error and successful indicated-speed readings. Same-recording real diagnostic frames are reported separately and do not select the checkpoint. Calibration and manual labels still have visual-estimation uncertainty.

Readings at most 2 km/h beyond a calibrated dial endpoint can be clamped to 0 or 180. Reports preserve the raw estimate and clamp flag. This fixed engineering tolerance was reviewed on this same recording; it is not evidence of independent accuracy.


In [ ]:
source_checkpoint=None
if TRAIN_MODE=="transfer":
    default_source=str(DRIVE_ROOT/"experiments/speedometer_arc_v2_augmented/runs/20261017_120108_959025_fee960/best_model.pt")
    explicit=SOURCE_CHECKPOINT if SOURCE_CHECKPOINT and SOURCE_CHECKPOINT!=default_source else ""
    try:
        source_checkpoint=nh.find_source_checkpoint(DRIVE_ROOT,SOURCE_SHA256,explicit)
    except FileNotFoundError:
        from google.colab import files
        print("Upload the approved epoch34 best_model.pt; exact SHA256 verification is required.")
        uploaded=files.upload()
        if len(uploaded)!=1:
            raise ValueError("Expected one source checkpoint")
        uploaded_name,uploaded_bytes=next(iter(uploaded.items()))
        if hashlib.sha256(uploaded_bytes).hexdigest()!=SOURCE_SHA256:
            raise ValueError("Uploaded checkpoint is not the approved epoch34 model")
        source_checkpoint=DRIVE_ROOT/"inputs/approved_epoch34/best_model.pt"
        source_checkpoint.parent.mkdir(parents=True,exist_ok=True)
        if source_checkpoint.exists() and nh.sha256(source_checkpoint)!=SOURCE_SHA256:
            raise ValueError("Existing source destination has a different checksum")
        if not source_checkpoint.exists():
            source_checkpoint.write_bytes(uploaded_bytes)
        nh.atomic_json(source_checkpoint.with_suffix(".complete.json"),{"checkpoint":source_checkpoint.name,"sha256":SOURCE_SHA256,"bytes":source_checkpoint.stat().st_size})
        del uploaded,uploaded_bytes
    print("Verified source:",source_checkpoint)

proposed_config=asdict(trainer.NeedleTrainConfig(data_dir=str(DATA_DIR),output_dir="",real_data_dir=str(REAL_DATA_DIR),include_real_training=True,
    epochs=int(EPOCHS),batch_size=int(BATCH_SIZE),decoder_lr=float(DECODER_LR),encoder_lr=float(ENCODER_LR),device="cuda",amp=True,num_workers=0,seed=42))
runtime={"torch":str(torch.__version__),"torchvision":str(torchvision.__version__),"numpy":str(np.__version__),"pillow":str(PIL.__version__),"opencv":str(cv2.__version__),"device":torch.cuda.get_device_name(0)}
EXPERIMENT_DIR,resolved_config,experiment_manifest=nh.prepare_experiment(DRIVE_ROOT,EXPERIMENT_NAME,TRAIN_MODE,proposed_config,
    code_commit=CODE_COMMIT,public_archive_sha256=PUBLIC_DATA_SHA256,private_archive_sha256=PRIVATE_DATA_SHA256,runtime=runtime)
CONFIG=trainer.NeedleTrainConfig(**resolved_config)
print("Experiment:",EXPERIMENT_DIR)
print("Fixed epoch target:",CONFIG.epochs,"| mode:",TRAIN_MODE)
print("Training: 400 synthetic + 10 approved real frames. Validation: 100 synthetic. Diagnostics: 9 real frames from the same recording.")
if TRAIN_MODE.startswith("resume_"):
    print("Saved configuration restored; edited epoch/LR settings above do not extend or alter this existing experiment.")


## 5. Train once and save every completed epoch
If interrupted, choose `resume_latest` and Run all. `resume_best` restores the entire selected checkpoint state, including optimizer and random state, and continues toward the same saved target. Incomplete or incompatible files never silently trigger fresh training.


In [ ]:
call_key=(str(EXPERIMENT_DIR),TRAIN_MODE,json.dumps(resolved_config,sort_keys=True))
if globals().get("_V3_ACTIVE_CALL_KEY")==call_key:
    raise RuntimeError("This training call is still active; wait for it to finish or interrupt it first.")
if globals().get("_V3_COMPLETED_CALL_KEY")==call_key:
    REPORT=_V3_LAST_REPORT
    print("This training call already completed; reusing its report without training twice.")
elif globals().get("_V3_STARTED_CALL_KEY")==call_key and not TRAIN_MODE.startswith("resume_"):
    raise RuntimeError("This initialization already started. Choose resume_latest and rerun setup; transfer/fresh cannot initialize twice.")
else:
    # A retry in resume mode selects a verified completed checkpoint again.
    _V3_STARTED_CALL_KEY=call_key
    _V3_ACTIVE_CALL_KEY=call_key
    try:
        REPORT=trainer.train(CONFIG,mode=TRAIN_MODE,source_checkpoint=source_checkpoint,search_root=EXPERIMENT_DIR/"runs")
    finally:
        _V3_ACTIVE_CALL_KEY=None
    _V3_LAST_REPORT=REPORT
    _V3_COMPLETED_CALL_KEY=call_key
print(json.dumps({key:REPORT.get(key) for key in ("best_epoch","completed_epoch","target_epoch","training_complete","best_model_path","latest_checkpoint_path","history_path")},indent=2))


## 6. Inspect the result and keep a compact export
All epoch states already live in Drive. The compact ZIP contains best inference weights and reports, not a second copy of every optimizer checkpoint. A lower synthetic validation error does not establish independent real-camera accuracy; inspect the real diagnostic predictions and keep new recordings untouched for later testing.


In [ ]:
print("Before adaptation:")
print(json.dumps(REPORT.get("before_metrics",{}),indent=2))
print("Selected best after adaptation:")
print(json.dumps(REPORT.get("after_metrics",{}),indent=2))
print("Same-recording real diagnostics:")
print(json.dumps(REPORT.get("real_diagnostics",{}),indent=2))
run_dir=Path(REPORT["run_dir"])
from real_adaptation_v3.render_diagnostics import render_real_comparison
from IPython.display import Image as DisplayImage
before_real_path=run_dir/"before_real_diagnostic_predictions.json"
after_real_path=run_dir/"real_diagnostic_predictions.json"
if REPORT["training_complete"] and before_real_path.is_file() and after_real_path.is_file():
    comparison=render_real_comparison(REAL_DATA_DIR,json.loads(before_real_path.read_text()),json.loads(after_real_path.read_text()),run_dir/"real_before_after.png")
    display(DisplayImage(filename=comparison))
    print("Private real comparison saved:",comparison)
else:
    print("Real comparison will be available when this saved epoch target completes.")
compact=EXPERIMENT_DIR/(run_dir.name+"_results.zip")
required_export={"best_model.pt","best_model.complete.json","training_report.json","experiment_config_snapshot.json"}
def verify_compact_archive(path):
    with zipfile.ZipFile(path) as archive:
        if archive.testzip() is not None or not required_export.issubset(archive.namelist()):
            raise ValueError("Compact export is incomplete or failed its CRC check")
if compact.exists():
    verify_compact_archive(compact)
else:
    temporary_export=compact.parent/(".results_"+uuid.uuid4().hex[:12]+".partial")
    try:
        with zipfile.ZipFile(temporary_export,"x",compression=zipfile.ZIP_DEFLATED,compresslevel=2) as archive:
            for path in sorted(run_dir.rglob("*")):
                if not path.is_file() or path.name.startswith("epoch_") or ".partial" in path.name:
                    continue
                if path.suffix.lower() in {".json",".csv",".png",".jpg",".py",".txt"} or path.name=="best_model.pt":
                    archive.write(path,path.relative_to(run_dir).as_posix())
            archive.write(EXPERIMENT_DIR/"experiment_config.json","experiment_config_snapshot.json")
        verify_compact_archive(temporary_export)
        os.replace(temporary_export,compact)
    finally:
        temporary_export.unlink(missing_ok=True)
print("Private Drive export:",compact)
print("SHA256:",nh.sha256(compact))
if DOWNLOAD_COMPACT_RESULTS:
    from google.colab import files
    files.download(str(compact))
